# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rasheed-hammad/machine-learning-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [40]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")
print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [41]:
!pip -q install duckdb huggingface_hub

In [42]:
import duckdb
from huggingface_hub import login, hf_hub_download

login(token=HF_TOKEN)

con = duckdb.connect()

print("Connected to Hugging Face and DuckDB.")

Connected to Hugging Face and DuckDB.


In [43]:
march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print(march_file)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [44]:
content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print(content_file)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_content.parquet


In [45]:
content_columns = con.execute(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{content_file}')
""").fetchdf()

display(content_columns)

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


## 1. My rule and its reason codes

Rule: Prioritize pages for refresh review when they are both stale and still visible in search. A page is considered stale when it has not been updated for at least 180 days, and visible when it has at least 500 impressions during the March 2026 panel. The score is the March impression count for pages meeting both conditions, so higher-visibility stale pages are reviewed first.

Reason code: stale_and_visible — the page is at least 180 days old since its last update and received at least 500 search impressions during March.

Other pages: Pages that do not meet both conditions receive not_prioritized and are assigned the monitor action.

In [46]:
# Load March 2026 performance and content metadata

march = con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM read_parquet('{march_file}')
""").fetchdf()

content = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        content_created_date,
        content_updated_date,
        content_type,
        is_published,
        is_deleted
    FROM read_parquet('{content_file}')
""").fetchdf()

print("March rows:", len(march))
print("Content metadata rows:", len(content))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows: 9841378
Content metadata rows: 519606


In [47]:
# Join March performance with content metadata
# Keep only published, non-deleted content with a known update date
# and prevent future metadata from leaking into the March snapshot.

df = march.merge(
    content,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

df["days_since_update"] = (
    df["report_date"] - df["content_updated_date"]
).dt.days

# CTR as a proportion
df["ctr"] = (
    df["gsc_clicks"] / df["gsc_impressions"]
).where(df["gsc_impressions"] > 0)

analysis_df = df[
    (df["is_published"] == True) &
    (df["is_deleted"] == False) &
    (df["content_updated_date"].notna()) &
    (df["days_since_update"] >= 0)
].copy()

print("Rows before filtering:", len(df))
print("Rows after filtering:", len(analysis_df))
print("Future-update rows excluded:",
      ((df["days_since_update"] < 0)).sum())

display(
    analysis_df[
        [
            "report_date",
            "client_hash_id",
            "content_hash_id",
            "days_since_update",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ctr"
        ]
    ].head()
)

Rows before filtering: 9841378
Rows after filtering: 1143001
Future-update rows excluded: 8672105


,report_date,client_hash_id,content_hash_id,days_since_update,gsc_impressions,gsc_clicks,gsc_avg_position,ctr
5095,2026-03-01,client_73cda7b4e4f265ea,content_8ef4a23c8dfdcbf8,4,1,0,52.0,0.0
5096,2026-03-01,client_73cda7b4e4f265ea,content_0ae27a6ddd60eeb5,4,10,0,1.0,0.0
5097,2026-03-01,client_73cda7b4e4f265ea,content_d3c09107f60a3cba,4,30,0,10.1,0.0
5098,2026-03-01,client_73cda7b4e4f265ea,content_7c3e8f9e825838cf,4,2,0,13.5,0.0
5099,2026-03-01,client_73cda7b4e4f265ea,content_f71331c9242a18cd,4,4,0,0.0,0.0


In [48]:
# Signal 1 — Staleness audit
import pandas as pd
analysis_df["staleness_bucket"] = pd.cut(
    analysis_df["days_since_update"],
    bins=[-1, 89, 179, 364, float("inf")],
    labels=["<90 days", "90–179 days", "180–364 days", "365+ days"]
)

staleness_table = (
    analysis_df
    .groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("gsc_impressions", "median"),
        median_ctr=("ctr", "median"),
        pct_visible_500=("gsc_impressions", lambda x: (x >= 500).mean() * 100)
    )
    .reset_index()
)

display(staleness_table)

,staleness_bucket,n,median_impressions,median_ctr,pct_visible_500
0,<90 days,929751,5.0,0.0,1.088894
1,90–179 days,132133,0.0,0.0,0.213421
2,180–364 days,81117,0.0,0.0,0.000000
3,365+ days,0,NaN,NaN,NaN


In [49]:
# Signal 2 — CTR vs position audit

position_df = analysis_df[
    (analysis_df["gsc_impressions"] >= 100) &
    (analysis_df["gsc_avg_position"].notna())
].copy()

position_df["position_bucket"] = pd.cut(
    position_df["gsc_avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1–3", "3–10", "10–20", "20+"]
)

position_table = (
    position_df
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("gsc_impressions", "median"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

display(position_table)

,position_bucket,n,median_impressions,median_ctr
0,1–3,17255,184.0,0.0
1,3–10,24998,183.0,0.0
2,10–20,3980,139.0,0.0
3,20+,25852,205.0,0.0


In [50]:
# Signal 3 — Search visibility / volume audit

volume_df = analysis_df.copy()

volume_df["volume_bucket"] = pd.cut(
    volume_df["gsc_impressions"],
    bins=[-1, 9, 99, 499, 999, float("inf")],
    labels=["0–9", "10–99", "100–499", "500–999", "1000+"]
)

volume_table = (
    volume_df
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_ctr=("ctr", "median"),
        median_position=("gsc_avg_position", "median")
    )
    .reset_index()
)

display(volume_table)

,volume_bucket,n,median_ctr,median_position
0,0–9,748958,0.000000,7.333333
1,10–99,321933,0.000000,7.670330
2,100–499,61704,0.000000,5.913042
3,500–999,7369,0.001285,5.284211
4,1000+,3037,0.000892,5.726576


In [51]:
page_df = (
    analysis_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "median"),
        days_since_update=("days_since_update", "max"),
        content_type=("content_type", "first")
    )
)

page_df["ctr"] = (
    page_df["clicks"] / page_df["impressions"]
).where(page_df["impressions"] > 0)

print("Unique pages:", len(page_df))
display(page_df.head())

Unique pages: 37229


,client_hash_id,content_hash_id,impressions,clicks,avg_position,days_since_update,content_type,ctr
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331,2,13.071429,34,keyword article,0.006042
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,0,9.375000,34,keyword article,0.000000
2,client_0797ff3a1fc9a6a5,content_0b33d8960857ad90,0,0,NaN,34,keyword article,NaN
3,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,145,0,8.666667,34,keyword article,0.000000
4,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,232,0,12.047619,34,keyword article,0.000000


In [52]:

# Section 1 — Rule definition

page_df["visible"] = (page_df["impressions"] >= 500).astype(int)
page_df["stale"] = (page_df["days_since_update"] >= 180).astype(int)

page_df["score"] = (
    page_df["stale"]
    * page_df["visible"]
    * page_df["impressions"]
)

page_df["reason_code"] = "not_prioritized"

page_df.loc[
    (page_df["stale"] == 1) & (page_df["visible"] == 1),
    "reason_code"
] = "stale_and_visible"

page_df["action"] = "monitor"

page_df.loc[
    page_df["reason_code"] == "stale_and_visible",
    "action"
] = "refresh_review"


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [53]:
import os

os.makedirs("work/outputs", exist_ok=True)

print("work/outputs folder is ready.")

work/outputs folder is ready.


In [54]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 — Build the ranked queue

# Rank highest-priority pages first
page_df = page_df.sort_values(
    ["score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

# Assign rank
page_df["rank"] = range(1, len(page_df) + 1)

# Create the final queue
queue = page_df[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "action",
        "reason_code",
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
        "days_since_update",
        "content_type"
    ]
].copy()

# Write the required CSV
output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Queue rows:", len(queue))
print("CSV written to:", output_path)

# Show Top 20
display(queue.head(20))

Queue rows: 37229
CSV written to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,score,action,reason_code,impressions,clicks,ctr,avg_position,days_since_update,content_type
0,1,client_c182d11e4862a37d,content_42ce26be1ec6be00,4411,refresh_review,stale_and_visible,4411,6,0.001360,4.227273,264,keyword article
1,2,client_c182d11e4862a37d,content_bea86ce3455100b0,3670,refresh_review,stale_and_visible,3670,1,0.000272,6.333333,232,keyword article
2,3,client_65de48885f4ef01b,content_eba53d72e18a9f93,734,refresh_review,stale_and_visible,734,2,0.002725,5.080000,231,keyword article
3,4,client_65de48885f4ef01b,content_c126a43258b574c3,592,refresh_review,stale_and_visible,592,0,0.000000,33.085973,231,keyword article
4,5,client_c182d11e4862a37d,content_5271624ae98fff86,550,refresh_review,stale_and_visible,550,3,0.005455,6.461538,231,keyword article
5,6,client_73cda7b4e4f265ea,content_f43118e089ecc69a,0,monitor,not_prioritized,139417,191,0.001370,5.021091,34,keyword article
6,7,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,0,monitor,not_prioritized,83834,1,0.000012,5.833333,34,keyword article
7,8,client_23a62021009f63c4,content_73aa61dcedebbf30,0,monitor,not_prioritized,80124,9,0.000112,45.257708,34,keyword article
8,9,client_73cda7b4e4f265ea,content_80eb6221de550658,0,monitor,not_prioritized,79766,175,0.002194,2.299586,34,keyword article
9,10,client_73cda7b4e4f265ea,content_ac7b77e81c53d636,0,monitor,not_prioritized,79651,165,0.002072,6.038301,34,keyword article


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [55]:
# Section 3 — Top-20 review

top20 = queue.head(20).copy()

top20["confidence_note"] = top20.apply(
    lambda row:
        "High confidence under the baseline rule: stale and visible."
        if row["reason_code"] == "stale_and_visible"
        else
        "Low confidence: included only because it has a zero score and was used to break the tie by impressions.",
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    lambda row:
        "The page may no longer need a refresh if the content was recently reviewed or updated outside the metadata captured here."
        if row["reason_code"] == "stale_and_visible"
        else
        "The page may be a poor review candidate because it is not stale under the rule, despite having high impressions.",
    axis=1
)

top20_review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

display(top20_review)


,rank,client_hash_id,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,client_c182d11e4862a37d,content_42ce26be1ec6be00,refresh_review,stale_and_visible,High confidence under the baseline rule: stale...,The page may no longer need a refresh if the c...
1,2,client_c182d11e4862a37d,content_bea86ce3455100b0,refresh_review,stale_and_visible,High confidence under the baseline rule: stale...,The page may no longer need a refresh if the c...
2,3,client_65de48885f4ef01b,content_eba53d72e18a9f93,refresh_review,stale_and_visible,High confidence under the baseline rule: stale...,The page may no longer need a refresh if the c...
3,4,client_65de48885f4ef01b,content_c126a43258b574c3,refresh_review,stale_and_visible,High confidence under the baseline rule: stale...,The page may no longer need a refresh if the c...
4,5,client_c182d11e4862a37d,content_5271624ae98fff86,refresh_review,stale_and_visible,High confidence under the baseline rule: stale...,The page may no longer need a refresh if the c...
5,6,client_73cda7b4e4f265ea,content_f43118e089ecc69a,monitor,not_prioritized,Low confidence: included only because it has a...,The page may be a poor review candidate becaus...
6,7,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,monitor,not_prioritized,Low confidence: included only because it has a...,The page may be a poor review candidate becaus...
7,8,client_23a62021009f63c4,content_73aa61dcedebbf30,monitor,not_prioritized,Low confidence: included only because it has a...,The page may be a poor review candidate becaus...
8,9,client_73cda7b4e4f265ea,content_80eb6221de550658,monitor,not_prioritized,Low confidence: included only because it has a...,The page may be a poor review candidate becaus...
9,10,client_73cda7b4e4f265ea,content_ac7b77e81c53d636,monitor,not_prioritized,Low confidence: included only because it has a...,The page may be a poor review candidate becaus...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

The weakest picks are ranks 6–20. They all have a score of 0 and were included in the top 20 only because impressions were used to break ties. They are not stale under the baseline rule, so they are not strong refresh candidates.

Some of these pages have high impressions and could still deserve investigation for other reasons, such as poor search position, but those signals are intentionally outside this baseline rule. This shows a limitation of the simple score: once the five stale-and-visible pages are ranked, the remaining queue is not meaningfully differentiated.

No product flags were used in the score. The baseline uses only March 2026 performance and content metadata available for the analysis. Future performance windows and starter outcome labels were not used. Rows where the content update date was after the report date were excluded before scoring, preventing future metadata from influencing an earlier observation.

This baseline is therefore a transparent decision-support rule, not a prediction of future performance.

In [56]:
# Section 4 — Weak picks + leakage check

weak_picks = top20[top20["action"] == "monitor"].copy()

print("Weak picks in Top-20:", len(weak_picks))
print("Refresh-review picks in Top-20:", (top20["action"] == "refresh_review").sum())

print("\nWeak picks:")
display(
    weak_picks[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "action",
            "reason_code",
            "impressions",
            "avg_position",
            "days_since_update"
        ]
    ]
)

print("\nLeakage checks:")
print("Starter outcome labels used:", False)
print("Future performance windows used:", False)
print("Product flags used:", False)
print("Future-update rows included in scoring:", False)

assert "trend_direction" not in page_df.columns
assert "trend_pct" not in page_df.columns
assert "is_declining_label" not in page_df.columns

print("\nLeakage check passed.")

Weak picks in Top-20: 15
Refresh-review picks in Top-20: 5

Weak picks:


,rank,client_hash_id,content_hash_id,action,reason_code,impressions,avg_position,days_since_update
5,6,client_73cda7b4e4f265ea,content_f43118e089ecc69a,monitor,not_prioritized,139417,5.021091,34
6,7,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,monitor,not_prioritized,83834,5.833333,34
7,8,client_23a62021009f63c4,content_73aa61dcedebbf30,monitor,not_prioritized,80124,45.257708,34
8,9,client_73cda7b4e4f265ea,content_80eb6221de550658,monitor,not_prioritized,79766,2.299586,34
9,10,client_73cda7b4e4f265ea,content_ac7b77e81c53d636,monitor,not_prioritized,79651,6.038301,34
10,11,client_23a62021009f63c4,content_49267c758cdcb3a8,monitor,not_prioritized,73768,32.770867,34
11,12,client_73cda7b4e4f265ea,content_e9f2d0579387d3c3,monitor,not_prioritized,73503,5.521801,34
12,13,client_73cda7b4e4f265ea,content_57dcb96896f9a33c,monitor,not_prioritized,70026,3.311777,34
13,14,client_23a62021009f63c4,content_b875a2f306635a58,monitor,not_prioritized,68657,31.694358,34
14,15,client_73cda7b4e4f265ea,content_f2388a4b87a3b1dc,monitor,not_prioritized,67646,5.231301,34



Leakage checks:
Starter outcome labels used: False
Future performance windows used: False
Product flags used: False
Future-update rows included in scoring: False

Leakage check passed.
